# Lab: Cortex Search Part 1

📚  In this lab you will learn and practice the following:

Part 1: Cortex Search Engine for unstructured data

   ❄️ Use the AI_PARSE_DOCUMENT function to extract the contents of PDF files

   ❄️ Use the SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER function to chunk the extracted text
   
   ❄️ Create a Cortex Search Service that runs on the table of text chunks
   
   ❄️ Query the Cortex Search Service using SEARCH_PREVIEW
   
   ❄️ Inspect vector embeddings using CORTEX_SEARCH_DATA_SCAN
   
   ❄️ Test the search service interactively in the Cortex Search Playground

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

If you find your queries are running for more than 5 minutes, cancel and come back and try them later.

If you find models that are deprecated, use CoCo to help you fix the issue by selecting a suitable model.

---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL statements. Select any query and ask *"Explain this SQL"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any function? Ask CoCo *"What does [function name] do?"* to get its syntax, supported options, and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.

---

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

## Introduction

Cortex Search provides low-latency, high-quality **fuzzy** search capabilities for Snowflake data, supporting various search experiences, including **Retrieval Augmented Generation (RAG)** applications using Large Language Models (LLMs). It offers a **hybrid vector and keyword search engine** that can be set up in minutes, without the need to manage embeddings, infrastructure, search tuning, or index maintenance. This enables users to focus on developing chat and search applications rather than managing infrastructure. Cortex Search is a fully managed service that automatically creates embeddings for your data and performs retrievals using a **hybrid approach: embeddings for semantic similarity, keyword search for lexical similarity, and semantic reranking for relevance**. We will perform some preprocessing steps before enabling Cortex Search.

### Getting started.

When using Cortex Search, it's important to consider the specific business challenge you're aiming to address and how the tool can help solve it. Do you need a search service, or do you need an LLM chatbot to answer questions from documents? Additionally, think about which data or documents you need to make available to Cortex Search for it to provide relevant answers through the service.

## Building a RAG pipeline for Travelbug

To minimize hallucinations (incorrect responses), large language models (LLMs) can be integrated with private datasets. A widely used method for reducing hallucinations without modifying the model itself (such as through fine-tuning) is the **Retrieval Augmented Generation** **(RAG)** framework. RAG **grounds** the model's responses by providing a set of relevant documents as context for the LLM during generation.

In this lab, we will build a RAG pipeline using Cortex Search and then use the **Playground** in **AI & ML > AI Studio > Search** to ask questions against the indexed data.

### Set up your current context for the role, database, schema and warehouse.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_genai_db'
print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r Set_up_your_current_context_for_the_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA raw;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: Cortex Search Part 1';
SHOW PARAMETERS LIKE 'query_tag' in session 
  ->> SELECT "value" AS query_tag FROM $1;

### Load the documents that are going to be used by the search service.

First, let's make sure we have some documents for the search service to run on. In our case, the Snowflake Education team has already loaded some PDF documents that contain details about the tour, and we cloned that stage in Lab 1. The documents can be found in the **{{user}}_GENAI_DB.RESOURCES.GENAI2DAY/SEARCH_DOCS**. 

Run the following commands to confirm you can see four PDF documents.

In [ ]:
%%sql -r Load_documents_list_stage_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA resources;

LIST @genai2day/search_docs;


### Create a table to store the chunked documents.

Next, we will create a new table called **PARSE_DOC** to parse the documents using **AI_PARSE_DOCUMENT** function. This table will include the relative path of each document, source_url, raw_text extracted from each PDF document.

In [ ]:
%%sql -r Create_a_table_to_store_the_chunked_sql
-- Create and populate PARSE_DOC in one step
CREATE TABLE IF NOT EXISTS {{user}}_genai_db.raw.PARSE_DOC AS
SELECT
    relative_path AS relative_path,
    GET_PRESIGNED_URL('@{{user}}_genai_db.resources.genai2day', relative_path) AS source_url,
    size AS size,
    TO_VARCHAR(
        AI_PARSE_DOCUMENT(
            TO_FILE('@{{user}}_genai_db.resources.genai2day', relative_path),
            {'mode': 'LAYOUT'}
        ):content
    ) AS raw_text
FROM DIRECTORY(@{{user}}_genai_db.resources.genai2day)
WHERE STARTSWITH(relative_path, 'search_docs/');

### Extract data into parse_doc table.

The statement below selects all PDF files from a specified storage location.

For each file, it retrieves:

*   **relative_path** : The file's path within the storage.
*   **scoped_url**     : for accessing the file.
*   **file_size**     : size of file

It then extracts text using **AI_PARSE_DOCUMENT**, which can operate in either **LAYOUT** mode (preserving formatting) or **OCR** mode (for scanned text).

The extracted text is converted into a string format.

Finally, the data is inserted into the **PARSE_DOC** table for downstream processing.

In [ ]:
%%sql -r Extract_data_into_parse_doc_table_sql
-- This INSERT is now handled by the CREATE TABLE IF NOT EXISTS above
-- Keeping this cell for reference - it will be skipped if table already exists
SELECT 'PARSE_DOC table ready' AS status;


### Query the contents of the PARSE_DOC table.

In [ ]:
%%sql -r Query_the_contents_of_the_PARSE_DOC_sql
SELECT *, LENGTH(raw_text) AS raw_text_size
FROM {{user}}_genai_db.raw.PARSE_DOC ;

### Create a table PARSE_DOC_CHUNKS to hold the chunked documents.

The **SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER** function is used to split the raw text into smaller chunks based on the following parameters:

The input text is treated as markdown, split into 1800-character chunks (~512 tokens) with a 300-character overlap to maintain context across chunks.

For clarity the ingestion pipeline steps are represented in multiple steps.

In [ ]:
%%sql -r Create_table_PARSE_DOC_CHUNKS_sql
-- Create table with explicit column list
CREATE TABLE IF NOT EXISTS {{user}}_genai_db.raw.parse_doc_chunks (
    doc_id,
    relative_path,
    category,
    chunk,
    source_url
) AS
SELECT
    UUID_STRING() AS doc_id,
    relative_path,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        TO_VARCHAR(c.value),
        ['sunsettour', 'mountainhike', 'winetour', 'balloonride', 'snorkeling']
    ):label::VARCHAR AS category,
    TO_VARCHAR(c.value) AS chunk,
    source_url
FROM {{user}}_genai_db.raw.parse_doc,
     LATERAL FLATTEN(
         input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
             raw_text,
             'markdown',
             1800,
             300
         )
     ) c;

### Insert chunked data into PARSE_DOC_CHUNKS table.

The query below inserts data into the **{{user}}_GENAI_DB.RAW.PARSE_DOC_CHUNKS** table.

Columns Selected:

**Relative_path**: From **{{user}}_GENAI_DB.RAW.PARSE_DOC**, directly.

**Category** :
Using Snowflake's **CORTEX.CLASSIFY_TEXT**, it classifies chunks of text into one of five categories (sunsettour, mountainhike, winetour, balloonride, snorkeling) by passing the text (from c.value) through a machine learning model.

**Chunk**:
The raw_text column is processed with **SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER** to split the text into chunks.

The chunks are cast to VARCHAR using **TO_VARCHAR(c.value)**.

**Source_url**: From **{{user}}_GENAI_DB.RAW.PARSE_DOC**, directly.

**LATERAL FLATTEN** is used to transform the result of the **SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER** function, which recursively splits the raw_text into 1800-character chunks (~512 tokens) with 300-character overlap.



In [ ]:
%%sql -r Insert_chunked_data_into_PARSE_DOC_sql
SELECT *, LENGTH(chunk) AS chunk_size
FROM {{user}}_genai_db.raw.PARSE_DOC_CHUNKS
ORDER BY relative_path;



### Create a Cortex Search Service that runs on the table with chunked PDFs.

Next, we need to build a search service using the **CREATE CORTEX SEARCH SERVICE** command. This service will generate embeddings based on the **chunk** column and retrieve information based on similarity, while using the "category" column as a filter. The search service will refresh every 30 minutes. The retrieved data will include the following:

- Doc ID
- Relative path
- Chunk 
- File URL
- Category

The search service is created with a straightforward SQL statement and automatically handles complex search index maintenance incrementally, eliminating the need for us to manage any infrastructure.

In [ ]:
%%sql -r Create_a_Cortex_Search_Service_that_sql
-- Create a Cortex Search Service 
CREATE OR REPLACE CORTEX SEARCH SERVICE {{user}}_genai_db.resources.travelbug_search
ON chunk
ATTRIBUTES category
WAREHOUSE = {{user}}_genai_wh
TARGET_LAG = '30 minutes'
EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT
        doc_id,
        chunk,
        relative_path,
        source_url,
        category::VARCHAR AS category
    FROM {{user}}_genai_db.raw.parse_doc_chunks
);

### Using search preview to confirm the search is working.

Use the **SNOWFLAKE.CORTEX.SEARCH_PREVIEW** function to test the search service with a sample query.

In [ ]:
%%sql -r Using_search_preview_to_confirm_the_sql
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    '{{user}}_genai_db.resources.travelbug_search',
    $${
        "query": "sunset tour activities",
        "columns": ["doc_id", "chunk", "category", "relative_path"],
        "limit": 3
    }$$
) AS search_results;

### Using the cortex_search_data_scan function to view vector embeddings.

Use the **CORTEX_SEARCH_DATA_SCAN** function to inspect the indexed data and vector embeddings generated by the search service.

In [ ]:
%%sql -r Using_the_cortex_search_data_scan_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA resources;
SELECT *
FROM
  TABLE (
    CORTEX_SEARCH_DATA_SCAN (
      SERVICE_NAME => 'TRAVELBUG_SEARCH'
    ));

## Test your Cortex Search Service in the Playground

Now that the Cortex Search service is created, you can test it interactively using the built-in **Search Playground** in Snowsight, no code required. The playground is showing you what your search service would return to an application or an AI agent.

### Steps:

1. In Snowsight, navigate to **AI & ML** > **Cortex AI** > **Search** from the left sidebar
2. You will see your **TRAVELBUG_SEARCH** service listed
3. Click on the service name to open the **Search Playground**
4. Select your Cortex Search service from the dropdown (if not already selected)
5. Type a question in the search box and press Enter

### Try these questions:

**Question 1:** What food will be provided during the balloon ride?
> A specific question about activity details.

**Question 2:** Are jackets needed for the sunset tour?
> Check clothing/equipment requirements.

**Question 3:** What is the recommended backpack size for the hike?
> Test retrieval of specific gear details.

**Question 4:** Which accessories are required for the hike?
> Another equipment-related query.

**Question 5:** Will there be a bike ride at the end of the hike?
> Test how the model handles questions where the answer may not exist in the source documents.

### How to interpret the results

**The Results (Left Side):** Each numbered result is a chunk — a section of a document that was indexed into your search service.

**The Settings (Right Side) — Three Scoring Techniques:**

Cortex Search uses three methods together to rank results, each weighted equally here (33.3% each):

| Technique | What it does |
|-----------|-------------|
| Text | Exact keyword matching — finds chunks containing the same words as your query |
| Vector | Semantic/meaning matching — finds chunks that mean the same thing, even with different words |
| Reranker | A second AI pass that re-scores the results for relevance after the first two methods run |

**Use reranker**  — this means the final order you see has already been re-evaluated for quality. This generally improves accuracy.

---

**Tip:** The Playground lets you experiment with different queries and see how the hybrid search engine retrieves and ranks relevant document chunks. Try rephrasing questions to see how the semantic understanding handles variations.

## 🎯 Challenge Questions

Test your understanding of Cortex Search concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "What type of search engine does Cortex Search use?", "options": ["A) A purely keyword-based search engine", "B) A hybrid search engine that combines lexical and vector search", "C) A traditional SQL-based search engine", "D) A graph-based search engine"], "hash": "a1c2dcb153d031b29e5487e111f1b14b"},
    {"q": "What technique is required to process larger documents with Cortex Search?", "options": ["A) Compression", "B) Indexing", "C) Chunking", "D) Caching"], "hash": "73a0389e59f5115d726479df5d596cac"},
    {"q": "What is the primary purpose of chunking documents before using Cortex Search?", "options": ["A) To reduce storage costs", "B) To process large documents into smaller pieces that fit within embedding model limits", "C) To speed up network transfers", "D) To enable parallel processing"], "hash": "210e288cfa658b44de52884dda74b8bd"},
    {"q": "Which function can be used to test a Cortex Search Service without building a full application?", "options": ["A) SEARCH_PREVIEW", "B) SEARCH_TEST", "C) SEARCH_QUERY", "D) SEARCH_VALIDATE"], "hash": "f3f4c69678bd78cd1d869905108d9d2a"},
    {"q": "What is a key benefit of using Cortex Search Service for building chatbots?", "options": ["A) Requires manual index updates", "B) Only supports exact keyword matching", "C) Requires complex infrastructure setup", "D) Automatically updates the search index when source data changes"], "hash": "a6f2b0216e20cc8ee962b2d1c7c6bbb0"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        escaped_opt = opt.replace("'", "''")
        result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
        feedback = result[0][0]
        is_correct = 'Correct' in feedback or '\u2705' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))

## Key Takeaways

❄️ Cortex Search enables you to build LLM-powered search applications by combining hybrid retrieval with language models.

❄️ Cortex Search service is simple to build using SQL and automatically updates the search index when source data changes.

❄️ The Cortex search engine is a hybrid search engine that uses lexical search, vector search, and semantic reranking for accurate retrieval of data.

❄️ To process larger documents with Cortex Search, the use of chunking is required.

❄️ AI_PARSE_DOCUMENT can be used to extract text content from PDF files for use with Cortex Search.

❄️ SEARCH_PREVIEW allows you to test and validate your Cortex Search service directly in SQL without building a full application.

❄️ The Cortex Search Playground (AI & ML > Cortex AI > Search) provides a no-code interface to interactively test queries against your search service.